In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r"C:\Users\dell\Downloads\online_retail_II.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
df.shape

(1067371, 8)

In [16]:
# 2 countries filter karne ke liye .isin() lagayein
selected_countries = ['France', 'United Kingdom']
france_df = df[df['Country'].isin(selected_countries)]

france_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [17]:
france_df.shape

(995660, 8)

In [19]:
selected_countries = ['United Kingdom']
uk_df = df[df['Country'].isin(selected_countries)]

uk_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [21]:
uk_df.Country.unique()

<ArrowStringArray>
['United Kingdom']
Length: 1, dtype: str

In [23]:
# Cleaning: Missing hatana aur cancelled orders exclude karna
clean_uk = uk_df.dropna(subset=['Description'])
clean_uk = clean_uk[~clean_uk['Invoice'].astype(str).str.startswith('C')]
clean_uk = clean_uk[clean_uk['Description'] != 'POSTAGE']

# Groupby: Har Invoice ke items ki list
transactions = clean_uk.groupby('Invoice')['Description'].apply(list).tolist()

In [26]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

# DataFrame banana (Memory bachane ke liye bool type use karein)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)
basket_df.head()

,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST,BROWN PIRATE TREASURE CHEST,Bank Charges,CAMPHOR WOOD PORTOBELLO MUSHROOM,CHERRY BLOSSOM DECORATIVE FLASK,...,wrongly coded 20713,wrongly coded 23343,wrongly coded-23343,wrongly marked,wrongly marked 23343,wrongly marked carton 22804,wrongly marked. 23343 in box,wrongly sold (22719) barcode,wrongly sold as sets,wrongly sold sets
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [27]:
basket_df.shape

(37461, 5651)

In [30]:
clean_uk.shape

(960218, 8)

In [33]:
from mlxtend.frequent_patterns import apriori, association_rules

# United Kingdom me transactions bohot zyada hain, isliye min_support 0.02 - 0.03 rakhein
frequent_items = apriori(basket_df, min_support=0.02, use_colnames=True)

# Association rules
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.5)
rules.sort_values(by='lift', ascending=False).head(10)
rules[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
1,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.025093,0.032968,0.020795,0.828723,25.137496
0,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({PINK REGENCY TEACUP AND SAUCER}),0.032968,0.025093,0.020795,0.630769,25.137496
2,frozenset({ROSES REGENCY TEACUP AND SAUCER }),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.033902,0.032968,0.024986,0.737008,22.355508
3,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.032968,0.033902,0.024986,0.757895,22.355508
10,frozenset({SWEETHEART CERAMIC TRINKET BOX}),frozenset({STRAWBERRY CERAMIC TRINKET BOX}),0.037586,0.057366,0.026454,0.703835,12.269135
8,frozenset({PAPER CHAIN KIT VINTAGE CHRISTMAS}),frozenset({PAPER CHAIN KIT 50'S CHRISTMAS }),0.036838,0.052348,0.021916,0.594928,11.364906
12,frozenset({WOODEN PICTURE FRAME WHITE FINISH}),frozenset({WOODEN FRAME ANTIQUE WHITE }),0.052161,0.053389,0.029097,0.557830,10.448437
11,frozenset({WOODEN FRAME ANTIQUE WHITE }),frozenset({WOODEN PICTURE FRAME WHITE FINISH}),0.053389,0.052161,0.029097,0.545000,10.448437
5,frozenset({LOVE BUILDING BLOCK WORD}),frozenset({HOME BUILDING BLOCK WORD}),0.044179,0.053416,0.022530,0.509970,9.547216
4,frozenset({HEART OF WICKER LARGE}),frozenset({HEART OF WICKER SMALL}),0.051627,0.056112,0.026427,0.511892,9.122742
